# Intermediate 13 — Multiple Linear Regression in Matrix Form

Practice notebook for the **"Multiple Linear Regression in Matrix Form"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


## Part 1 — The model and the OLS closed form

The PDF writes the multiple linear regression model as

\[
y \;=\; X\beta + \varepsilon,
\]

with $y\in\mathbb{R}^{n}$ the response, $X\in\mathbb{R}^{n\times p}$ the design
matrix (first column usually ones, for the intercept), $\beta\in\mathbb{R}^{p}$
the coefficients, and $\varepsilon\in\mathbb{R}^{n}$ the noise.

The least squares estimator solves

$$
\hat{\beta} \;=\; \arg\min_{\beta} \lVert y - X\beta\rVert^{2},
$$

and, when $X^{\top}X$ is invertible, has the closed form

$$
\hat{\beta} \;=\; (X^{\top}X)^{-1} X^{\top} y.
$$

Below we simulate data from a known $\beta$, recover the coefficients with the
matrix formula, and cross-check against `np.linalg.lstsq` (which uses a numerically
stable QR/SVD route to the same solution).


In [ ]:
# A known "true" model with three predictors + intercept
rng = np.random.default_rng(7)
n, p = 200, 4
beta_true = np.array([1.5, 2.0, -1.0, 0.5])  # intercept + 3 slopes

X = rng.normal(size=(n, p))
X[:, 0] = 1.0                              # intercept column
sigma = 1.0
y = X @ beta_true + rng.normal(0.0, sigma, size=n)

# OLS closed form:  beta_hat = (X^T X)^{-1} X^T y
XtX = X.T @ X
Xty = X.T @ y
beta_hat = np.linalg.inv(XtX) @ Xty
print("beta_true :", np.round(beta_true, 4))
print("beta_hat   :", np.round(beta_hat, 4))
print("abs diff   :", np.round(np.abs(beta_hat - beta_true), 4))

# Cross-check against np.linalg.lstsq (QR-based, numerically stable)
beta_lstsq, *_ = np.linalg.lstsq(X, y, rcond=None)
print("beta_lstsq :", np.round(beta_lstsq, 4))
print("max |beta_hat - beta_lstsq| :", round(float(np.max(np.abs(beta_hat - beta_lstsq))), 12))

# Residuals and R^2
resid = y - X @ beta_hat
SSE  = float(resid @ resid)                 # sum of squared residuals
SST  = float(((y - y.mean()) ** 2).sum())
R2   = 1.0 - SSE / SST
print("SSE :", round(SSE, 4), "| R^2 :", round(R2, 4))


## Part 2 — Residual variance and standard errors of the coefficients

Under the Gauss–Markov assumptions $E[\varepsilon]=0$ and
$\operatorname{Cov}(\varepsilon)=\sigma^{2} I_{n}$, the PDF gives

$$
\operatorname{Cov}(\hat{\beta}) \;=\; \sigma^{2} (X^{\top} X)^{-1}.
$$

The noise level $\sigma^{2}$ is unknown and is estimated unbiasedly by the residual
variance

$$
S^{2} \;=\; \frac{1}{n-p}\lVert y - X\hat{\beta}\rVert^{2}.
$$

Plugging in, the estimated covariance of the coefficients is
$\widehat{\operatorname{Cov}}(\hat{\beta}) = S^{2}(X^{\top}X)^{-1}$, and the
standard errors are the square roots of its diagonal.

We compute them and check the $S^{2}$ estimate against the true $\sigma^{2}$, then run a
short simulation to confirm that $\hat{\beta}$ is **unbiased** and that the empirical
variance of $\hat{\beta}_j$ across replications matches the theoretical
$\sigma^{2}[(X^{\top}X)^{-1}]_{jj}$.


In [ ]:
# Residual variance and standard errors on the single fit from Part 1
dof = n - p
S2 = SSE / dof
cov_beta = S2 * np.linalg.inv(XtX)
se_beta = np.sqrt(np.diag(cov_beta))
print("sigma^2 true :", round(sigma ** 2, 4))
print("S^2 estimate :", round(S2, 4))
print()
print("coef   beta_hat    std_err")
for j, name in enumerate(["intercept", "x1", "x2", "x3"]):
    print(f"{name:9s} {beta_hat[j]:+.4f}   {se_beta[j]:.4f}")

# --- Simulation: check unbiasedness and that Cov(hat beta) = sigma^2 (X^T X)^{-1}
R = 5000
rng = np.random.default_rng(123)
# Keep the SAME X for every replication (fixed design) so theory applies exactly.
betas = np.empty((R, p))
for r in range(R):
    yr = X @ beta_true + rng.normal(0.0, sigma, size=n)
    betas[r] = np.linalg.inv(XtX) @ (X.T @ yr)

emp_mean = betas.mean(axis=0)
emp_cov  = np.cov(betas, rowvar=False, ddof=1)
theo_cov = (sigma ** 2) * np.linalg.inv(XtX)
print()
print("Empirical mean of hat beta :", np.round(emp_mean, 4))
print("True beta                   :", np.round(beta_true, 4))
print()
print("Empirical Var(hat beta_j)   :", np.round(np.diag(emp_cov), 5))
print("Theory  sigma^2 (XtX)^-1 jj :", np.round(np.diag(theo_cov), 5))


## Part 3 — Gauss–Markov: OLS is BLUE

The Gauss–Markov theorem says: among all **linear unbiased** estimators of
$C\beta$ (for any fixed $C$), $C\hat{\beta}$ has the smallest variance.
Equivalently, OLS is the **Best Linear Unbiased Estimator (BLUE)**.

A biased estimator can beat OLS on *mean squared error* by trading a little bias for a
big variance reduction. The classic example is **ridge regression**,

$$
\hat{\beta}_{\lambda} \;=\; (X^{\top}X + \lambda I)^{-1} X^{\top}y,
$$

which shrinks coefficients toward zero. Below we keep the **same fixed design** $X$ as in
Parts 1–2 and compare, across many replications:

- the OLS variance (matches $\sigma^{2}(X^{\top}X)^{-1}$ exactly in theory),
- the ridge variance (smaller),
- the ridge **bias** (non-zero — ridge is not unbiased, so Gauss–Markov does not apply),
- the ridge **MSE** (bias$^{2}+$var; for large enough $\lambda$ this can be *below* OLS).

This is the sense in which OLS is "best" only *within the unbiased* class.


In [ ]:
def ols_beta(yv):
    return np.linalg.inv(XtX) @ (X.T @ yv)

def ridge_beta(yv, lam):
    # Note: penalize only slopes, not the intercept (standardize convention).
    pen = lam * np.eye(p)
    pen[0, 0] = 0.0
    return np.linalg.inv(XtX + pen) @ (X.T @ yv)

R = 8000
lam = 5.0
rng = np.random.default_rng(321)
B_ols   = np.empty((R, p))
B_ridge = np.empty((R, p))
for r in range(R):
    yr = X @ beta_true + rng.normal(0.0, sigma, size=n)
    B_ols[r]   = ols_beta(yr)
    B_ridge[r] = ridge_beta(yr, lam)

# Per-coefficient bias, variance, MSE
print(f"lambda = {lam}")
print(f"{'coef':9s} {'OLS var':>9s} {'Ridge var':>10s} {'Ridge bias^2':>13s} {'Ridge MSE':>10s} {'OLS MSE':>9s}")
for j, name in enumerate(["intercept", "x1", "x2", "x3"]):
    ols_var   = B_ols[:, j].var(ddof=1)
    ridge_var = B_ridge[:, j].var(ddof=1)
    ridge_b2  = (B_ridge[:, j].mean() - beta_true[j]) ** 2
    ridge_mse  = ridge_var + ridge_b2
    ols_mse   = ols_var + (B_ols[:, j].mean() - beta_true[j]) ** 2
    print(f"{name:9s} {ols_var:9.5f} {ridge_var:10.5f} {ridge_b2:13.5f} {ridge_mse:10.5f} {ols_mse:9.5f}")

# Visualize the OLS-vs-ridge variance trade-off for one coefficient (x1, index 1)
j = 1
plt.figure()
plt.hist(B_ols[:, j], bins=60, alpha=0.6, density=True, label="OLS")
plt.hist(B_ridge[:, j], bins=60, alpha=0.6, density=True, label=f"Ridge $\\lambda$={lam:g}")
plt.axvline(beta_true[j], color="k", linestyle="--", label="true $\\beta_{x1}$")
plt.xlabel("$\\hat\\beta_{x1}$")
plt.ylabel("density")
plt.title("OLS wider & centered; ridge tighter & biased — the BLUE trade-off")
plt.legend()
plt.tight_layout()
plt.show()


## Part 4 — Inference: t-statistics and the overall F-test

With the stronger Gaussian assumption $\varepsilon\sim\mathcal{N}(0, \sigma^{2}I_{n})$
the PDF notes we get exact sampling distributions: $\hat{\beta}$ is multivariate
normal and $S^{2}\sim\sigma^{2}\chi^{2}_{n-p}/(n-p)$ independent of $\hat{\beta}$.

- **Individual coefficients** use $t$-statistics
  $T_j = \hat{\beta}_j / \operatorname{SE}(\hat{\beta}_j) \sim t_{n-p}$ under
  $H_0:\beta_j=0$.
- **Joint / overall significance** uses an $F$-statistic built from the residual sum of
  squares of the **full** model ($\mathrm{SSE}_{\mathrm{full}}$) and a **reduced**
  model that drops the predictors being tested ($\mathrm{SSE}_{\mathrm{red}}$):

$$
F \;=\; \frac{(\mathrm{SSE}_{\mathrm{red}} - \mathrm{SSE}_{\mathrm{full}}) / q}
              {\mathrm{SSE}_{\mathrm{full}} / (n-p)}
\;\sim\; F_{q,\,n-p}
\quad\text{under } H_0,
$$

where $q$ is the number of restrictions. The "overall regression" F-test drops *all* slopes
(intercept-only reduced model), so $q = p - 1$.

We build $F$ by hand from residual sums of squares and compare it to the value obtained
two ways: from the `scipy.stats.f` distribution (p-value) and from the identity
$F = \frac{R^{2}/(p-1)}{(1-R^{2})/(n-p)}$ for the overall test.


In [ ]:
# Reuse the single fit from Part 1/2 (beta_hat, resid, SSE, R2 already in scope).
# --- t-statistics for individual coefficients ---
t_stats = beta_hat / se_beta
p_vals_t = 2.0 * stats.t.sf(np.abs(t_stats), df=dof)
print("coef       beta_hat    std_err     t         p-value")
for j, name in enumerate(["intercept", "x1", "x2", "x3"]):
    print(f"{name:9s} {beta_hat[j]:+.4f}   {se_beta[j]:.4f}   {t_stats[j]:+.4f}   {p_vals_t[j]:.4f}")

# --- Overall F-test (drop ALL slopes; intercept-only reduced model) ---
SSE_full = SSE                       # from Part 1
SSE_red  = float(((y - y.mean()) ** 2).sum())  # intercept-only model residuals
q = p - 1                            # number of restrictions (slopes)
F_stat = ((SSE_red - SSE_full) / q) / (SSE_full / dof)
p_val_F = stats.f.sf(F_stat, dfn=q, dfd=dof)

# Identity cross-check: F = (R^2 / (p-1)) / ((1-R^2) / (n-p))
F_from_R2 = (R2 / (p - 1)) / ((1.0 - R2) / dof)
print()
print("SSE_full :", round(SSE_full, 4))
print("SSE_red  :", round(SSE_red, 4))
print("F_stat   :", round(F_stat, 4))
print("F from R^2 identity :", round(F_from_R2, 4))
print("p-value (F-test)    :", f"{p_val_F:.3e}")

# --- A partial F-test: do x2 and x3 jointly matter? Drop them, keep intercept + x1. ---
X_red = X[:, [0, 1]]                 # intercept + x1 only
beta_red, *_ = np.linalg.lstsq(X_red, y, rcond=None)
SSE_red2 = float(((y - X_red @ beta_red) ** 2).sum())
q2 = 2                               # we dropped 2 predictors (x2, x3)
F_partial = ((SSE_red2 - SSE_full) / q2) / (SSE_full / dof)
p_partial = stats.f.sf(F_partial, dfn=q2, dfd=dof)
print()
print("Partial F (x2,x3 jointly):")
print("  F_stat   :", round(F_partial, 4))
print("  p-value  :", f"{p_partial:.3e}")


## Part 5 — The F-statistic's sampling distribution under $H_0$

The theory says that **if the null is true** (all tested slopes are zero), the
hand-built $F$-statistic follows an $F_{q,\,n-p}$ distribution exactly. We verify this
by simulation: fix a design $X$, simulate $y$ with the tested slopes set to zero,
compute $F$ each time, and compare the empirical distribution of $F$ to the
$\operatorname{F}(q, n-p)$ density from `scipy.stats.f`.

We also check the Type-I error rate: at level $\alpha=0.05$, the fraction of
replications with $F > F_{0.95,\,q,\,n-p}$ should be close to $0.05$.

**Your turn:** change `q_slopes` (the number of truly-zero slopes being tested) and
confirm the distribution still matches. Then make one slope *non-zero* and watch the
distribution shift right and the rejection rate climb above $\alpha$.


In [ ]:
# Design with an intercept and 3 noise predictors whose slopes we will set to 0
n_sim, p_sim = 120, 4
Xs = rng.normal(size=(n_sim, p_sim))
Xs[:, 0] = 1.0
q_slopes = p_sim - 1            # testing all 3 slopes jointly
dof_sim = n_sim - p_sim
sigma_sim = 2.0

# True beta: intercept only, all slopes = 0  (so H0 is TRUE)
beta_null = np.zeros(p_sim)
beta_null[0] = 5.0

R = 20000
rng = np.random.default_rng(999)
F_sim = np.empty(R)
for r in range(R):
    ys = Xs @ beta_null + rng.normal(0.0, sigma_sim, size=n_sim)
    # Full model
    bf, *_ = np.linalg.lstsq(Xs, ys, rcond=None)
    sse_full = float(((ys - Xs @ bf) ** 2).sum())
    # Reduced: intercept only
    sse_red = float(((ys - ys.mean()) ** 2).sum())
    F_sim[r] = ((sse_red - sse_full) / q_slopes) / (sse_full / dof_sim)

# Compare to F(q, n-p) density
grid = np.linspace(0.01, 6.0, 400)
plt.figure()
plt.hist(F_sim, bins=80, density=True, alpha=0.6, label=f"simulated $F$ ($R$={R})")
plt.plot(grid, stats.f.pdf(grid, dfn=q_slopes, dfd=dof_sim),
         "r-", lw=2, label=f"$F_{{{q_slopes},{dof_sim}}}$ density")
plt.xlabel("$F$-statistic")
plt.ylabel("density")
plt.title(r"Overall $F$-statistic under $H_0$ matches $F_{q,\,n-p}$ exactly")
plt.legend()
plt.tight_layout()
plt.show()

# Type-I error check at alpha = 0.05
F_crit = stats.f.ppf(0.95, dfn=q_slopes, dfd=dof_sim)
rejection_rate = np.mean(F_sim > F_crit)
print(f"F critical (0.95) : {F_crit:.4f}")
print(f"rejection rate    : {rejection_rate:.4f}   (target 0.05)")
print(f"mean F (sim)      : {F_sim.mean():.4f}   (theory {stats.f.mean(dfn=q_slopes, dfd=dof_sim):.4f})")


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. Write down the normal equations $X^{\top}X\hat{\beta} = X^{\top}y$ and show that, when $X^{\top}X$ is invertible, the minimizer of $\lVert y-X\beta\rVert^{2}$ is $\hat{\beta}=(X^{\top}X)^{-1}X^{\top}y$.

2. For a fixed design $X$ with $n=100$, $p=4$ and $\sigma^{2}=4$, suppose $(X^{\top}X)^{-1}_{11}=0.05$ (the entry for the intercept). Compute $\operatorname{Var}(\hat{\beta}_0)$, $\operatorname{SE}(\hat{\beta}_0)$, and the 95% confidence interval for $\beta_0$ if $\hat{\beta}_0=2.0$.

3. State the Gauss–Markov theorem precisely (assumptions, claim, and the class of estimators OLS is best within). Then explain in one sentence why ridge regression with $\lambda>0$ is *not* a counterexample to the theorem.

4. Given $\mathrm{SSE}_{\mathrm{full}}=80$, $\mathrm{SSE}_{\mathrm{red}}=200$, $q=3$ restrictions, and $n-p=40$ residual degrees of freedom in the full model, compute the $F$-statistic and the approximate p-value using $F_{3,40}$.

5. In Part 5 we set all slopes to zero and the simulated $F$-statistic followed $F_{q,n-p}$. Explain what happens to the distribution of $F$ and to the rejection rate at level $0.05$ if exactly one of the tested slopes is non-zero. Why does the test eventually reject with probability approaching 1 as $n\to\infty$?


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** Expand the objective $S(\beta)=\lVert y-X\beta\rVert^{2}=(y-X\beta)^{\top}(y-X\beta) = y^{\top}y - 2\beta^{\top}X^{\top}y + \beta^{\top}X^{\top}X\beta$. Setting the gradient to zero gives the normal equations $-2X^{\top}y+2X^{\top}X\beta=0$, i.e. $X^{\top}X\hat{\beta}=X^{\top}y$. If $X^{\top}X$ is invertible, multiply both sides by $(X^{\top}X)^{-1}$ to get $\hat{\beta}=(X^{\top}X)^{-1}X^{\top}y$. The Hessian $2X^{\top}X$ is positive definite, so this is the unique minimizer.

**2.** $\operatorname{Var}(\hat{\beta}_0)=\sigma^{2}\,(X^{\top}X)^{-1}_{11}=4\cdot 0.05=0.20$. So $\operatorname{SE}(\hat{\beta}_0)=\sqrt{0.20}\approx 0.447$. With $n-p=96$ degrees of freedom, $t_{0.975,\,96}\approx 1.985$, so the 95% CI is $\hat{\beta}_0 \pm 1.985\cdot 0.447 \approx 2.0 \pm 0.887$, i.e. $[1.113,\,2.887]$.

**3.** Under $E[\varepsilon]=0$ and $\operatorname{Cov}(\varepsilon)=\sigma^{2}I_{n}$ (homoskedastic, uncorrelated errors), for any fixed matrix $C$ the estimator $C\hat{\beta}$ has the smallest variance among all linear unbiased estimators of $C\beta$. In other words, OLS is the Best Linear Unbiased Estimator (BLUE). Ridge with $\lambda>0$ gives $\hat{\beta}_{\lambda}=(X^{\top}X+\lambda I)^{-1}X^{\top}y=Ay$ for a matrix $A$ that depends on $\lambda$ but not on $y$, so it is linear in $y$, but $E[\hat{\beta}_{\lambda}]=A X\beta\neq\beta$ in general — it is biased, so it falls outside the unbiased class the theorem compares over.

**4.** $F = \frac{(\mathrm{SSE}_{\mathrm{red}} - \mathrm{SSE}_{\mathrm{full}})/q}{\mathrm{SSE}_{\mathrm{full}}/(n-p)} = \frac{(200-80)/3}{80/40} = \frac{40}{2} = 20$. Under $F_{3,40}$, $P(F>20)\approx 5{\times}10^{-9}$, so we reject $H_0$ at any conventional level: the dropped predictors jointly matter.

**5.** With one true slope $\beta_j\neq 0$, the numerator $\mathrm{SSE}_{\mathrm{red}}-\mathrm{SSE}_{\mathrm{full}}$ is inflated by the squared omitted signal (roughly $\beta_j^{2}$ times the residual variance of $X_j$ after projecting out the rest of $X$), so the $F$-statistic shifts to the right of $F_{q,n-p}$ and the rejection rate at $0.05$ rises above $0.05$ — i.e. the test gains power. As $n\to\infty$, $\hat{\beta}_j\to\beta_j\neq 0$ and $\operatorname{SE}(\hat{\beta}_j)\to 0$, so $F\to\infty$ in probability; the p-value goes to 0 and the test rejects with probability tending to 1 (consistent power).

</details>
